In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [17]:
import os
DATA_DIR = os.path.join(os.getcwd(), "Embedding", "5")

# Load embeddings (shape: n_samples x 300)
questions_emb  = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
answerkeys_emb = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))

# Metadata langsung dari pickle
meta_path = os.path.join(DATA_DIR, 'final_metadata.pkl')
if os.path.exists(meta_path):
    metadata = pd.read_pickle(meta_path)
else:
    raise FileNotFoundError("aug_metadata.pkl tidak ditemukan di folder notebook.")

assert len(metadata) == len(answers_emb), \
    f"Mismatch: metadata={len(metadata)}, answers_emb={len(answers_emb)}"

print("=== Hasil Load ===")
print(f"questions_emb  : {questions_emb.shape}")
print(f"answerkeys_emb : {answerkeys_emb.shape}")
print(f"answers_emb    : {answers_emb.shape}")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())


=== Hasil Load ===
questions_emb  : (17, 135, 300)
answerkeys_emb : (17, 90, 300)
answers_emb    : (2351, 85, 300)

Metadata       : 2351 rows
Kolom metadata : ['IDJwb', 'IDPSJ', 'grade', 'psj_idx', 'is_synthetic']

IDPSJ unik     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

Distribusi grade:
grade
1     246
2     229
3     218
4     220
5     208
6     233
7     225
8     247
9     217
10    308
Name: count, dtype: int64


In [18]:
# ── Analisis Distribusi per IDPSJ untuk Persiapan Mixup ──────────────────────

# Pivot: baris=IDPSJ, kolom=grade, nilai=jumlah sampel
pivot = (metadata.groupby(['IDPSJ', 'grade'])
                 .size()
                 .unstack(fill_value=0)
                 .reindex(columns=range(1, 11), fill_value=0))

print("=" * 70)
print("Jumlah sampel per grade per IDPSJ")
print("=" * 70)
print(pivot.to_string())

# Statistik ringkas per IDPSJ
stats = pd.DataFrame({
    'total'   : pivot.sum(axis=1),
    'max_cls' : pivot.max(axis=1),
    'min_cls' : pivot[pivot > 0].min(axis=1),
    'n_kelas' : (pivot > 0).sum(axis=1),
})
stats['median_cls'] = pivot.replace(0, np.nan).median(axis=1)
stats['target_1:2'] = (stats['max_cls'] / 2).apply(np.ceil).astype(int)
stats['target_1:3'] = (stats['max_cls'] / 3).apply(np.ceil).astype(int)

print("\n" + "=" * 70)
print("Statistik per IDPSJ")
print("=" * 70)
print(stats.to_string())

# Ringkasan global
global_max = pivot.max().max()
global_target_half = int(np.ceil(global_max / 2))
print(f"\nNilai kelas terbanyak secara global : {global_max}")
print(f"Target rasio 1:2 (max/2)            : {global_target_half}")
print(f"Target flat 10                      : 10")
print(f"\nKelas yang AKAN di-augmentasi (< target 1:2) per IDPSJ:")
for psj in pivot.index:
    row = pivot.loc[psj]
    local_max = row.max()
    tgt = int(np.ceil(local_max / 2))
    kurang = row[(row > 0) & (row < tgt)]
    if not kurang.empty:
        detail = ", ".join([f"grade {g}:{cnt}→{tgt}" for g, cnt in kurang.items()])
        print(f"  IDPSJ {psj} (target={tgt}): {detail}")


Jumlah sampel per grade per IDPSJ
grade  1   2   3   4   5   6   7   8   9   10
IDPSJ                                        
1      19  19  19  19  19  25  19  19  19  37
2       8   8  15   8   8   8   8  10   8   8
3       5   5   5   5   5  10   5   6   5   7
4       9   9   9  15   9  10  10   9   9  17
5      17   9   9   9   9   9   9   9   9   9
6      22  11  11  11  11  11  11  11  11  11
7       9   9   9   9   9  10   9   9   9  17
8      27  27  27  27  27  27  27  27  27  53
9      24  24  24  24  24  24  24  24  24  47
10     12  12  12  12  12  12  18  23  19  21
11     21  42  21  21  21  21  21  21  21  22
12     11  11  11  11  11  11  21  11  11  11
13     10  10  10  10  10  11  10  20  10  10
14      9   8   8   8   8  12   8  15   8   9
15     11   6   9   9   6  10   6   9   6  10
16      5   5   5   8   5   8   5  10   7   5
17     27  14  14  14  14  14  14  14  14  14

Statistik per IDPSJ
       total  max_cls  min_cls  n_kelas  median_cls  target_1:2  target

In [14]:
# ── Mixup Augmentasi pada Embedding 300D ─────────────────────────────────────
# Strategi : intra-class Mixup per IDPSJ, hanya pada answers_emb
# questions_emb & answerkeys_emb TIDAK di-mix karena nilainya identik
# per IDPSJ — shape (17, seq, 300), diindeks via psj-1, bukan sample index.
# Target   : ceil(max_cls_per_IDPSJ / 2) untuk setiap kelas minoritas
# λ        : Beta(α=0.4) di-clip ke [0.5, 1.0] agar label tidak melompat

RNG   = np.random.default_rng(seed=42)
ALPHA = 0.4

syn_answers = []
syn_meta    = []

for psj in sorted(metadata['IDPSJ'].unique()):
    mask_psj = metadata['IDPSJ'] == psj
    psj_idx  = metadata.index[mask_psj].to_numpy()
    psj_meta = metadata.loc[mask_psj].reset_index(drop=True)
    local_max = psj_meta['grade'].value_counts().max()
    target    = int(np.ceil(local_max / 2))

    for grade in range(1, 11):
        cls_local_idx = psj_meta.index[psj_meta['grade'] == grade].to_numpy()
        n_cls = len(cls_local_idx)
        if n_cls == 0 or n_cls >= target:
            continue  # tidak perlu augmentasi

        n_needed = target - n_cls
        global_cls_idx = psj_idx[cls_local_idx]  # indeks ke answers_emb

        # Sampel pasangan secara acak (dengan penggantian)
        idx_i = RNG.choice(global_cls_idx, size=n_needed, replace=True)
        idx_j = RNG.choice(global_cls_idx, size=n_needed, replace=True)

        # λ ~ Beta(α, α) di-clip ke [0.5, 1.0]
        lam = RNG.beta(ALPHA, ALPHA, size=n_needed)
        lam = np.clip(lam, 0.5, 1.0)

        # Reshape λ agar bisa broadcast dengan dimensi embedding (n, ..., 300)
        lam_bc = lam.reshape((n_needed,) + (1,) * (answers_emb.ndim - 1))

        # Mixup hanya pada answers_emb
        syn_answers.append(lam_bc * answers_emb[idx_i] + (1 - lam_bc) * answers_emb[idx_j])

        # Label sintetis (intra-class → grade tetap sama)
        syn_grade = np.full(n_needed, grade, dtype=int)
        for k in range(n_needed):
            syn_meta.append({
                'IDJwb'        : f'syn_{psj}_{grade}_{k}',
                'IDPSJ'        : psj,
                'grade'        : syn_grade[k],
                'is_synthetic' : True,
            })

# ── Gabungkan data asli + sintetis ───────────────────────────────────────────
if syn_answers:
    aug_answers_emb = np.vstack([answers_emb] + syn_answers)
    aug_metadata    = pd.concat([metadata, pd.DataFrame(syn_meta)],
                                ignore_index=True)
else:
    aug_answers_emb = answers_emb.copy()
    aug_metadata    = metadata.copy()

# questions & answerkeys tetap per IDPSJ (tidak diubah oleh Mixup)
aug_questions_emb  = questions_emb   # (17, 135, 300)
aug_answerkeys_emb = answerkeys_emb  # (17, 90, 300)

n_syn = len(aug_metadata) - len(metadata)
print(f"Data asli      : {len(metadata)}")
print(f"Data sintetis  : {n_syn}")
print(f"Total          : {len(aug_metadata)}")
print(f"\naug_answers_emb    : {aug_answers_emb.shape}")
print(f"aug_questions_emb  : {aug_questions_emb.shape}")
print(f"aug_answerkeys_emb : {aug_answerkeys_emb.shape}")
print(f"\nDistribusi grade setelah augmentasi:")
print(aug_metadata['grade'].value_counts().sort_index())

Data asli      : 1591
Data sintetis  : 760
Total          : 2351

aug_answers_emb    : (2351, 85, 300)
aug_questions_emb  : (17, 135, 300)
aug_answerkeys_emb : (17, 90, 300)

Distribusi grade setelah augmentasi:
grade
1     246
2     229
3     218
4     220
5     208
6     233
7     225
8     247
9     217
10    308
Name: count, dtype: int64


In [15]:
# ── Analisis Distribusi per IDPSJ untuk Persiapan Mixup ──────────────────────

# Pivot: baris=IDPSJ, kolom=grade, nilai=jumlah sampel
pivot = (aug_metadata.groupby(['IDPSJ', 'grade'])
                 .size()
                 .unstack(fill_value=0)
                 .reindex(columns=range(1, 11), fill_value=0))

print("=" * 70)
print("Jumlah sampel per grade per IDPSJ")
print("=" * 70)
print(pivot.to_string())

# Statistik ringkas per IDPSJ
stats = pd.DataFrame({
    'total'   : pivot.sum(axis=1),
    'max_cls' : pivot.max(axis=1),
    'min_cls' : pivot[pivot > 0].min(axis=1),
    'n_kelas' : (pivot > 0).sum(axis=1),
})
stats['median_cls'] = pivot.replace(0, np.nan).median(axis=1)
stats['target_1:2'] = (stats['max_cls'] / 2).apply(np.ceil).astype(int)
stats['target_1:3'] = (stats['max_cls'] / 3).apply(np.ceil).astype(int)

print("\n" + "=" * 70)
print("Statistik per IDPSJ")
print("=" * 70)
print(stats.to_string())

# Ringkasan global
global_max = pivot.max().max()
global_target_half = int(np.ceil(global_max / 2))
print(f"\nNilai kelas terbanyak secara global : {global_max}")
print(f"Target rasio 1:2 (max/2)            : {global_target_half}")
print(f"Target flat 10                      : 10")
print(f"\nKelas yang AKAN di-augmentasi (< target 1:2) per IDPSJ:")
for psj in pivot.index:
    row = pivot.loc[psj]
    local_max = row.max()
    tgt = int(np.ceil(local_max / 2))
    kurang = row[(row > 0) & (row < tgt)]
    if not kurang.empty:
        detail = ", ".join([f"grade {g}:{cnt}→{tgt}" for g, cnt in kurang.items()])
        print(f"  IDPSJ {psj} (target={tgt}): {detail}")


Jumlah sampel per grade per IDPSJ
grade  1   2   3   4   5   6   7   8   9   10
IDPSJ                                        
1      19  19  19  19  19  25  19  19  19  37
2       8   8  15   8   8   8   8  10   8   8
3       5   5   5   5   5  10   5   6   5   7
4       9   9   9  15   9  10  10   9   9  17
5      17   9   9   9   9   9   9   9   9   9
6      22  11  11  11  11  11  11  11  11  11
7       9   9   9   9   9  10   9   9   9  17
8      27  27  27  27  27  27  27  27  27  53
9      24  24  24  24  24  24  24  24  24  47
10     12  12  12  12  12  12  18  23  19  21
11     21  42  21  21  21  21  21  21  21  22
12     11  11  11  11  11  11  21  11  11  11
13     10  10  10  10  10  11  10  20  10  10
14      9   8   8   8   8  12   8  15   8   9
15     11   6   9   9   6  10   6   9   6  10
16      5   5   5   8   5   8   5  10   7   5
17     27  14  14  14  14  14  14  14  14  14

Statistik per IDPSJ
       total  max_cls  min_cls  n_kelas  median_cls  target_1:2  target

In [16]:
# ── Simpan hasil akhir ───────────────────────────────────────────────────────
# answers_emb : simpan penuh karena setiap sampel unik
# questions & answerkeys : sudah satu baris per IDPSJ, simpan langsung
#   → rekonstruksi di training menggunakan kolom psj_idx di metadata

# Buat mapping IDPSJ → indeks unik (konsisten dengan urutan questions_emb)
idpsj_sorted = sorted(aug_metadata['IDPSJ'].unique())
idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
final_metadata = aug_metadata.copy()
final_metadata['psj_idx'] = final_metadata['IDPSJ'].map(idpsj_to_idx)

# Simpan
np.save(os.path.join(DATA_DIR, 'final_answers_emb.npy'),    aug_answers_emb)
np.save(os.path.join(DATA_DIR, 'final_questions_emb.npy'),  aug_questions_emb)
np.save(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'), aug_answerkeys_emb)
final_metadata.to_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))

print("File tersimpan:")
print(f"  final_answers_emb.npy    {aug_answers_emb.shape}  (per sampel)")
print(f"  final_questions_emb.npy  {aug_questions_emb.shape}  (per IDPSJ)")
print(f"  final_answerkeys_emb.npy {aug_answerkeys_emb.shape}  (per IDPSJ)")
print(f"  final_metadata.pkl       {len(final_metadata)} rows  (+ kolom psj_idx)")

File tersimpan:
  final_answers_emb.npy    (2351, 85, 300)  (per sampel)
  final_questions_emb.npy  (17, 135, 300)  (per IDPSJ)
  final_answerkeys_emb.npy (17, 90, 300)  (per IDPSJ)
  final_metadata.pkl       2351 rows  (+ kolom psj_idx)
